# DS2002 · Walmart Warm-Up SKU Mess

**Lab — 2026-10-16 · Fall 2026**  

---

## Lab 08 — Walmart Warm-Up (SKU Mess)

This lab uses the **real midterm data** in `../06-midterm-walmart/data/`. It's a warm-up so you hit the ground running on the project. On Kaggle/Colab, upload the data folder or adjust the path.

*If the file isn't found, the cell falls back to a small inline sample so the notebook still runs.*

In [7]:
import pandas as pd
PATH = 'product_catalog.csv'
try:
    cat = pd.read_csv(PATH)
except FileNotFoundError:
    from io import StringIO
    cat = pd.read_csv(StringIO('sku,product_name,category\nPT-12,pop-tarts strawberry,Breakfast Foods\nSKU#459812,Strawberry Pop-Tart,Breakfast\nPOPTART-STRAW,Pop-Tarts Strawberry,Breakfast'))
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack


### Q1 — How many rows look like Pop-Tart products?

Filter on `product_name` containing `pop`, case-insensitively. Print the count and the matching rows.

In [9]:
# TODO (hint: .str.lower().str.contains('pop'))
contains_pop = cat[cat['product_name'].str.lower().str.contains('pop', na=False)]
print(len(contains_pop))
contains_pop


4


,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


I filtered the orignial table to only rows that contain the string 'pop' in the product_name column, looking at all the entries in the column as lowercase strings.

### Q2 — Which SKUs and names all mean *strawberry* Pop-Tart?

List them. Then note which Pop-Tart rows are **not** strawberry — those must stay separate, and saying so is part of the answer.

In [10]:
# TODO
contains_strawberry = contains_pop[contains_pop['product_name'].str.lower().str.contains('strawberry', na=False)]
contains_strawberry

,sku,product_name,category,subcategory,unit_cost,supplier,package_size
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack


**Pop-Tart rows that are a different product:** All of the Pop-Tart rows are strawberry Pop-Tarts

### Q3 — Build the mapping

A dict from each variant SKU to one canonical SKU. Then apply it in a way that leaves unmapped SKUs as themselves rather than turning them into `NaN` — that single mistake silently deletes most of the catalog.

In [11]:
# TODO
mapping = {
    'PT-12': 'POPTART-STRAW',
    'SKU#459812': 'POPTART-STRAW',
    'PT_STRAWBERRY': 'POPTART-STRAW'
}

cat['canonical'] = cat['sku'].map(mapping).fillna(cat['sku'])
cat.head()

,sku,product_name,category,subcategory,unit_cost,supplier,package_size,canonical
0,PT-12,pop-tarts strawberry,Breakfast Foods,NaN,1.93,Johnson & Johnson,4-pack,POPTART-STRAW
1,SKU#459812,Strawberry Pop-Tart,Breakfast,Pop-Tarts,2.26,Roku Inc,4pk,POPTART-STRAW
2,POPTART-STRAW,Pop-Tarts Strawberry,Breakfast,NaN,2.11,Sara Lee,6 pack,POPTART-STRAW
3,PT_STRAWBERRY,pop-tarts strawberry,breakfast,Pop-Tarts,NaN,Campbell's,4 pack,POPTART-STRAW
4,WTR-24,BOTTLED WATER 24PK,Emergency Supplies,Bottled,2.64,Sara Lee,1 pack,WTR-24


mapping maps each of the other 3 sku values for strawberry poptart to the 4th value, and then I created a new column that contains the mapped value for sku, and fills with the original sku value for any rows that are not in the map.

### Q4 — Normalize `category`

Strip and title-case, then show the distinct values before and after with counts. Note any pair that is still two labels for one thing — `Breakfast` and `Breakfast Foods` are not the same string but they are the same category.

In [12]:
# TODO
print(cat['category'].value_counts(dropna=False))
cat['category'] = cat['category'].str.strip().str.title()
print(cat['category'].value_counts(dropna=False))

category
Emergency              5
Emergency Supplies     3
EMERGENCY              3
ENTERTAINMENT          3
GROCERY                3
Breakfast              2
entertainment          2
household              2
ELECTRONICS            2
health                 2
Baby                   2
Electronics            1
ALCOHOL                1
Alcoholic Beverages    1
breakfast              1
Breakfast Foods        1
Groceries              1
Grocery                1
electronics            1
snacks                 1
grocery                1
HOUSEHOLD              1
Snacks                 1
Household              1
baby                   1
Baby Products          1
emergency              1
Name: count, dtype: int64
category
Emergency              9
Entertainment          5
Grocery                5
Electronics            4
Household              4
Emergency Supplies     3
Breakfast              3
Baby                   3
Snacks                 2
Health                 2
Breakfast Foods        1
Alcoho

Still 2 labels for one thing: Emergency/Emergency Supplies; Breakfast/Breakfast Foods; Alcohol/Alcoholic Beverages; Grocery/Groceries; Baby/Baby Products

### Q5 — Find the fragmentation you have not spotted yet

Hand-scanning a catalog does not scale. **TODO:** build a candidate list programmatically — normalize each product name (lowercase, strip punctuation, sort the words) and group by that, then show any group holding more than one SKU.

This is the workflow from Monday's lecture, and it is how you will actually approach the full catalog on the midterm.

In [18]:
# TODO
import string

def normalize(product):
    if pd.isna(product):
        return ""
    cleaned = str(product).lower().translate(str.maketrans('', '', string.punctuation))
    result = sorted(cleaned.split())
    return " ".join(result)

cat['product_name_normalized'] = cat['product_name'].apply(normalize)

grouped_by_product = cat.groupby('product_name_normalized')

In [17]:
multiple_skus = grouped_by_product.filter(lambda g: g['sku'].nunique() > 1)
multiple_skus_grouped = multiple_skus.groupby('product_name_normalized')
for name, group in multiple_skus_grouped:
    print(name)

24pack beer light
4pack aa batteries
board game monopoly
bread white
butter peanut
candles emergency
canned soup
cards playing
chips pack variety
duty heavy tarp
flashlight led
paper towels
poptarts strawberry


These product names all contain multiple SKUs. I found this by creating a function that take each value in a column, normalizes it, and adds the normalized name to an empty string, while keeping the empty string for any row with an NA value. I then applied this function to the product_name column in the original dataset, and grouped by normalized product name. I filtered to groups with multiple unique SKUs to find product names with multiple SKUs.

### Q6 — Prove the mapping is safe

**TODO:** assert that applying your mapping produced no nulls in the canonical column and did not change the number of rows. Consolidation renames; it never adds or removes.

In [20]:
# TODO
assert cat['product_name_normalized'].notna().all()
assert len(cat) == 45
print("mapping successful")

mapping successful


### Write-up

**a)** Name two product families beyond Pop-Tarts that appear fragmented, and how you would group them.

**b)** Where does consolidation stop being a data-cleaning task and become a business decision? Give one example from this catalog where you genuinely are not sure, and say who you would ask.

Canned Soup and White Bread both appear fragmented. I would group them by supplier, which is essentially the brand name, since these products are most logically subcategorized by brand.

Consolidation becomes a business decision when deciding in what cases it is convenient to group items, and what cases it causes more inconvenience by concealing important data. For example, I would look into grouping snacks and grocery, but it is possible that these items are stored in different areas and grouping them would make it more difficult to keep track of them. It would make the most sense to ask a store manager who is more familiar with the layout and practical needs.